In [1]:
# Idiosyncratic risk collection + dynamic asset allocation

import pandas as pd
import numpy as np
import yfinance as yf
import os
import matplotlib.pyplot as plt
import requests

data_path = os.path.expanduser("~/Desktop/SentriVaR-500/data")

# Load saved data
copula_scores = pd.read_csv(f"{data_path}/copula_risk_scores.csv",
                             index_col="Date", parse_dates=True)

TICKERS = ["AAPL", "MSFT", "GOOGL", "JPM", "SOXX"]

print("Loaded")
print(f"Copula risk scores: {copula_scores.shape}")

Loaded
Copula risk scores: (2105, 5)


In [2]:
# Step 1. Collect analyst rating changes
def get_analyst_risk(ticker):
    """
    Idiosyncratic risk score from analyst rating changes.
    - Higher downgrade ratio = higher risk
    """
    try:
        stock = yf.Ticker(ticker)
        rec = stock.recommendations

        if rec is None or rec.empty:
            print(f"  {ticker}: no rating data → default 0.5")
            return 0.5

        # Use last 3 months of ratings
        recent = rec.tail(20)

        # Map grades to numeric scores
        grade_map = {
            "Strong Buy": -1.0, "Buy": -0.5, "Outperform": -0.3,
            "Neutral": 0.0,  "Hold": 0.0,  "Market Perform": 0.0,
            "Underperform": 0.5, "Sell": 0.8, "Strong Sell": 1.0
        }

        scores = []
        for _, row in recent.iterrows():
            to_grade = str(row.get("To Grade", "Neutral"))
            from_grade = str(row.get("From Grade", "Neutral"))

            to_score   = grade_map.get(to_grade, 0.0)
            from_score = grade_map.get(from_grade, 0.0)

            # Positive if downgraded, negative if upgraded
            scores.append(to_score - from_score)

        if not scores:
            return 0.5

        # Rescale -1..1 to 0..1
        avg = np.mean(scores)
        return round(max(0, min(1, (avg + 1) / 2)), 4)

    except Exception as e:
        print(f"  {ticker} error: {e}")
        return 0.5

# Fetch for all tickers
print("Fetching analyst ratings...")
analyst_scores = {}
for ticker in TICKERS:
    score = get_analyst_risk(ticker)
    analyst_scores[ticker] = score
    print(f"  {ticker}: {score:.4f} ({'risky' if score > 0.5 else 'safe'})")

Fetching analyst ratings...
  AAPL: 0.5000 (safe)
  MSFT: 0.5000 (safe)
  GOOGL: 0.5000 (safe)
  JPM: 0.5000 (safe)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SOXX"}}}


  SOXX: no rating data → default 0.5
  SOXX: 0.5000 (safe)


In [3]:
stock = yf.Ticker("AAPL")
rec = stock.recommendations
print(type(rec))
print(rec)

<class 'pandas.DataFrame'>
  period  strongBuy  buy  hold  sell  strongSell
0     0m          6   22    17     1           1
1    -1m          6   22    16     1           2
2    -2m          7   23    15     1           2
3    -3m          7   25    14     1           1


In [4]:
## Adjusted to match the actual data format
def get_analyst_risk(ticker):
    """
    Idiosyncratic risk score from aggregate analyst ratings.
    - Higher Sell/StrongSell ratio = higher risk
    """
    try:
        stock = yf.Ticker(ticker)
        rec = stock.recommendations

        if rec is None or rec.empty:
            print(f"  {ticker}: no data → default 0.5")
            return 0.5

        # Use most recent month (0m) only
        latest = rec[rec["period"] == "0m"].iloc[0]

        strong_buy  = latest["strongBuy"]
        buy         = latest["buy"]
        hold        = latest["hold"]
        sell        = latest["sell"]
        strong_sell = latest["strongSell"]
        total       = strong_buy + buy + hold + sell + strong_sell

        if total == 0:
            return 0.5

        # Weighted average: strongBuy=0, buy=0.25, hold=0.5, sell=0.75, strongSell=1.0
        weighted = (
            strong_buy  * 0.0  +
            buy         * 0.25 +
            hold        * 0.5  +
            sell        * 0.75 +
            strong_sell * 1.0
        ) / total

        return round(weighted, 4)

    except Exception as e:
        print(f"  {ticker} error: {e}")
        return 0.5

# Fetch for all tickers
print("Fetching analyst ratings...")
analyst_scores = {}
for ticker in TICKERS:
    score = get_analyst_risk(ticker)
    analyst_scores[ticker] = score
    print(f"  {ticker}: {score:.4f} ({'risky' if score > 0.5 else 'safe'})")

Fetching analyst ratings...
  AAPL: 0.3351 (safe)
  MSFT: 0.2098 (safe)
  GOOGL: 0.2227 (safe)
  JPM: 0.3333 (safe)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SOXX"}}}


  SOXX: no data → default 0.5
  SOXX: 0.5000 (safe)


In [5]:
# Step 2. Collect earnings surprises
def get_earnings_risk(ticker):
    """
    Idiosyncratic risk score from recent earnings surprises.
    - More misses = higher risk
    """
    try:
        stock = yf.Ticker(ticker)
        earnings = stock.earnings_dates

        if earnings is None or earnings.empty:
            print(f"  {ticker}: no earnings data → default 0.5")
            return 0.5

        # Check for EPS Surprise % column
        if "EPS Surprise %" not in earnings.columns:
            print(f"  {ticker}: no EPS Surprise column → default 0.5")
            return 0.5

        # Use last 4 quarters
        recent = earnings.dropna(subset=["EPS Surprise %"]).head(4)
        if recent.empty:
            return 0.5

        # Surprise % → risk score
        # Positive (beat) = low risk, negative (miss) = high risk
        surprises = recent["EPS Surprise %"].values
        avg_surprise = np.mean(surprises)

        # Normalize -50% to +50% into 0..1 (closer to 1 = worse miss)
        score = max(0, min(1, 0.5 - avg_surprise / 100))
        return round(score, 4)

    except Exception as e:
        print(f"  {ticker} error: {e}")
        return 0.5

# Fetch for all tickers
print("Fetching earnings surprises...")
earnings_scores = {}
for ticker in TICKERS:
    score = get_earnings_risk(ticker)
    earnings_scores[ticker] = score
    print(f"  {ticker}: {score:.4f} ({'miss risk' if score > 0.5 else 'beat, stable'})")

Fetching earnings surprises...
  AAPL: no EPS Surprise column → default 0.5
  AAPL: 0.5000 (beat, stable)
  MSFT: no EPS Surprise column → default 0.5
  MSFT: 0.5000 (beat, stable)
  GOOGL: no EPS Surprise column → default 0.5
  GOOGL: 0.5000 (beat, stable)
  JPM: no EPS Surprise column → default 0.5
  JPM: 0.5000 (beat, stable)


SOXX: No earnings dates found, symbol may be delisted


  SOXX: no earnings data → default 0.5
  SOXX: 0.5000 (beat, stable)


In [6]:
# Check actual column names
stock = yf.Ticker("AAPL")
earnings = stock.earnings_dates
print(earnings.columns.tolist())
print(earnings.head())

['EPS Estimate', 'Reported EPS', 'Surprise(%)']
                           EPS Estimate  Reported EPS  Surprise(%)
Earnings Date                                                     
2026-07-30 16:00:00-04:00          1.89           NaN          NaN
2026-04-30 16:00:00-04:00          1.94          2.01         3.46
2026-01-29 16:00:00-05:00          2.67          2.84         6.34
2025-10-30 16:00:00-04:00          1.77          1.85         4.52
2025-07-31 16:00:00-04:00          1.43          1.57        10.12


In [7]:
# Fixed column name
def get_earnings_risk(ticker):
    try:
        stock = yf.Ticker(ticker)
        earnings = stock.earnings_dates

        if earnings is None or earnings.empty:
            return 0.5

        # Last 4 quarters, drop NaN
        recent = earnings.dropna(subset=["Surprise(%)"]).head(4)
        if recent.empty:
            return 0.5

        avg_surprise = recent["Surprise(%)"].mean()

        # beat (positive) = low risk, miss (negative) = high risk
        score = max(0, min(1, 0.5 - avg_surprise / 100))
        return round(score, 4)

    except Exception as e:
        print(f"  {ticker} error: {e}")
        return 0.5

print("Fetching earnings surprises...")
earnings_scores = {}
for ticker in TICKERS:
    score = get_earnings_risk(ticker)
    earnings_scores[ticker] = score
    print(f"  {ticker}: {score:.4f} ({'miss risk' if score > 0.5 else 'beat, stable'})")

SOXX: No earnings dates found, symbol may be delisted


Fetching earnings surprises...
  AAPL: 0.4389 (beat, stable)
  MSFT: 0.4207 (beat, stable)
  GOOGL: 0.1721 (beat, stable)
  JPM: 0.4538 (beat, stable)
  SOXX: 0.5000 (beat, stable)


In [8]:
# Step 3. Collect SEC EDGAR insider transaction data
def get_insider_risk(ticker):
    """
    Idiosyncratic risk score from SEC EDGAR insider transactions.
    - More insider selling = higher risk
    """
    try:
        # Query SEC by CIK
        search_url = f"https://efts.sec.gov/LATEST/search-index?q=%22{ticker}%22&dateRange=custom&startdt=2026-01-01&enddt=2026-07-08&forms=4"
        headers = {"User-Agent": "SentriVaR research@example.com"}
        response = requests.get(search_url, headers=headers, timeout=10)

        if response.status_code != 200:
            return 0.5

        data = response.json()
        hits = data.get("hits", {}).get("hits", [])

        if not hits:
            return 0.5

        # Estimate buy/sell ratio from Form 4 filings
        buy_count  = 0
        sell_count = 0

        for hit in hits[:20]:  # last 20 filings
            source = hit.get("_source", {})
            form_type = source.get("file_type", "")
            if form_type == "4":
                # Infer buy/sell from entity name
                entity = source.get("entity_name", "").lower()
                if "purchase" in entity or "buy" in entity:
                    buy_count += 1
                else:
                    sell_count += 1

        total = buy_count + sell_count
        if total == 0:
            return 0.5

        # Higher sell ratio = higher risk
        sell_ratio = sell_count / total
        return round(sell_ratio, 4)

    except Exception as e:
        print(f"  {ticker} error: {e}")
        return 0.5

print("Fetching insider transactions...")
insider_scores = {}
for ticker in TICKERS:
    score = get_insider_risk(ticker)
    insider_scores[ticker] = score
    print(f"  {ticker}: {score:.4f} ({'sell-heavy' if score > 0.5 else 'buy-heavy'})")

Fetching insider transactions...
  AAPL: 1.0000 (sell-heavy)
  MSFT: 1.0000 (sell-heavy)
  GOOGL: 1.0000 (sell-heavy)
  JPM: 1.0000 (sell-heavy)
  SOXX: 0.5000 (buy-heavy)


In [9]:
# Check the actual SEC response
ticker = "AAPL"
search_url = f"https://efts.sec.gov/LATEST/search-index?q=%22{ticker}%22&dateRange=custom&startdt=2026-01-01&enddt=2026-07-08&forms=4"
headers = {"User-Agent": "SentriVaR research@example.com"}
response = requests.get(search_url, headers=headers, timeout=10)

data = response.json()
hits = data.get("hits", {}).get("hits", [])

print(f"Total hits: {len(hits)}")
if hits:
    print("\nFirst item keys:")
    print(hits[0]["_source"].keys())
    print("\nFirst item content:")
    print(hits[0]["_source"])

Total hits: 26

First item keys:
dict_keys(['ciks', 'period_ending', 'file_num', 'display_names', 'xsl', 'schema_version', 'sequence', 'root_forms', 'file_date', 'biz_states', 'sics', 'form', 'adsh', 'film_num', 'biz_locations', 'file_type', 'file_description', 'inc_states', 'items'])

First item content:
{'ciks': ['0002100523', '0000320193'], 'period_ending': '2026-05-08', 'file_num': ['001-36743'], 'display_names': ['Borders Ben  (CIK 0002100523)', 'Apple Inc.  (CIK 0000320193)'], 'xsl': 'xslF345X06', 'schema_version': 'X0609', 'sequence': 1, 'root_forms': ['4'], 'file_date': '2026-05-12', 'biz_states': ['CA'], 'sics': ['3571'], 'form': '4', 'adsh': '0001140361-26-020871', 'film_num': ['26970260'], 'biz_locations': ['', 'Cupertino, CA'], 'file_type': '4', 'file_description': 'FORM 4', 'inc_states': ['', 'CA'], 'items': []}


In [10]:
# Switch to yfinance's insider_transactions instead
def get_insider_risk(ticker):
    """
    Idiosyncratic risk score from yfinance insider transactions.
    - More insider selling = higher risk
    """
    try:
        stock = yf.Ticker(ticker)
        insider = stock.insider_transactions

        if insider is None or insider.empty:
            print(f"  {ticker}: no insider data → default 0.5")
            return 0.5

        print(f"  {ticker} columns: {insider.columns.tolist()}")
        print(insider.head(3))
        return 0.5

    except Exception as e:
        print(f"  {ticker} error: {e}")
        return 0.5

# Test on AAPL only
get_insider_risk("AAPL")

  AAPL columns: ['Shares', 'Value', 'URL', 'Text', 'Insider', 'Position', 'Transaction', 'Start Date', 'Ownership']
   Shares    Value URL                             Text            Insider  \
0     116  34236.0      Sale at price 295.14 per share.        BORDERS BEN   
1   30104      NaN                                       NEWSTEAD JENNIFER   
2     240      NaN                                             BORDERS BEN   

          Position Transaction Start Date Ownership  
0          Officer             2026-06-16         D  
1  General Counsel             2026-06-15         D  
2          Officer             2026-06-15         D  


0.5

In [11]:
def get_insider_risk(ticker):
    """
    Idiosyncratic risk score from yfinance insider transactions.
    - Higher sale ratio = higher risk
    """
    try:
        stock = yf.Ticker(ticker)
        insider = stock.insider_transactions

        if insider is None or insider.empty:
            return 0.5

        # Parse Sale/Purchase from the Text column
        insider["is_sale"] = insider["Text"].str.contains("Sale", case=False, na=False)
        insider["is_buy"]  = insider["Text"].str.contains("Purchase|Buy", case=False, na=False)

        # Weight by transaction value (larger deals matter more)
        sale_value = insider[insider["is_sale"]]["Value"].fillna(0).sum()
        buy_value  = insider[insider["is_buy"]]["Value"].fillna(0).sum()
        total_value = sale_value + buy_value

        if total_value == 0:
            # Fall back to counting transactions
            sale_count = insider["is_sale"].sum()
            buy_count  = insider["is_buy"].sum()
            total = sale_count + buy_count
            if total == 0:
                return 0.5
            return round(sale_count / total, 4)

        # Higher sell value ratio = higher risk
        return round(sale_value / total_value, 4)

    except Exception as e:
        print(f"  {ticker} error: {e}")
        return 0.5

print("Fetching insider transactions...")
insider_scores = {}
for ticker in TICKERS:
    score = get_insider_risk(ticker)
    insider_scores[ticker] = score
    print(f"  {ticker}: {score:.4f} ({'sell-heavy' if score > 0.5 else 'buy-heavy'})")

Fetching insider transactions...
  AAPL: 1.0000 (sell-heavy)
  MSFT: 0.9878 (sell-heavy)
  GOOGL: 1.0000 (sell-heavy)
  JPM: 1.0000 (sell-heavy)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SOXX"}}}


  SOXX: 0.5000 (buy-heavy)


In [12]:
def get_insider_risk(ticker):
    """
    Measure risk from insider selling acceleration.
    - Recent selling spiking relative to the past = higher risk
    """
    try:
        stock = yf.Ticker(ticker)
        insider = stock.insider_transactions

        if insider is None or insider.empty:
            return 0.5

        insider["Start Date"] = pd.to_datetime(insider["Start Date"])
        insider["is_sale"] = insider["Text"].str.contains("Sale", case=False, na=False)

        # Split by recency
        cutoff = pd.Timestamp.today() - pd.DateOffset(months=3)
        recent = insider[insider["Start Date"] >= cutoff]
        older  = insider[insider["Start Date"] < cutoff]

        recent_sale = recent[recent["is_sale"]]["Value"].fillna(0).sum()
        recent_total = recent["Value"].fillna(0).sum()
        older_sale = older[older["is_sale"]]["Value"].fillna(0).sum()
        older_total = older["Value"].fillna(0).sum()

        recent_ratio = recent_sale / recent_total if recent_total > 0 else 0.5
        older_ratio  = older_sale  / older_total  if older_total  > 0 else 0.5

        # Higher recent sell ratio relative to the past = higher risk
        delta = recent_ratio - older_ratio
        score = max(0, min(1, 0.5 + delta * 0.5))
        return round(score, 4)

    except Exception as e:
        print(f"  {ticker} error: {e}")
        return 0.5

print("Fetching insider transactions...")
insider_scores = {}
for ticker in TICKERS:
    score = get_insider_risk(ticker)
    insider_scores[ticker] = score
    print(f"  {ticker}: {score:.4f} ({'selling accelerating' if score > 0.5 else 'selling stable'})")

Fetching insider transactions...
  AAPL: 0.5000 (selling stable)
  MSFT: 0.5063 (selling accelerating)
  GOOGL: 0.2500 (selling stable)
  JPM: 0.5055 (selling accelerating)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SOXX"}}}


  SOXX: 0.5000 (selling stable)


In [13]:
# Step 4. Combine idiosyncratic risk signals
idiosyncratic_risk = {}
for ticker in TICKERS:
    # Weights: earnings (40%) + analyst (30%) + insider (30%)
    score = (
        earnings_scores[ticker]  * 0.40 +
        analyst_scores[ticker]   * 0.30 +
        insider_scores[ticker]   * 0.30
    )
    idiosyncratic_risk[ticker] = round(score, 4)

print("Idiosyncratic risk score per ticker:")
print("-" * 40)
for ticker, score in idiosyncratic_risk.items():
    bar = "█" * int(score * 20)
    print(f"  {ticker:5s}: {score:.4f}  {bar}")

print("\nRiskiest ticker:", max(idiosyncratic_risk, key=idiosyncratic_risk.get))
print("Safest ticker:", min(idiosyncratic_risk, key=idiosyncratic_risk.get))

Idiosyncratic risk score per ticker:
----------------------------------------
  AAPL : 0.4261  ████████
  MSFT : 0.3831  ███████
  GOOGL: 0.2106  ████
  JPM  : 0.4332  ████████
  SOXX : 0.5000  ██████████

Riskiest ticker: SOXX
Safest ticker: GOOGL


In [14]:
# Step 5. Dynamic asset allocation
# Combine portfolio-level risk with per-ticker idiosyncratic risk

# Current portfolio-level risk
portfolio_risk = copula_scores["copula_risk_score"].iloc[-1]
current_regime = copula_scores["regime"].iloc[-1]

print(f"Current portfolio risk: {portfolio_risk:.4f}")
print(f"Current regime: {current_regime} ({'Normal' if current_regime==0 else 'Elevated' if current_regime==1 else 'Crisis'})")

def dynamic_allocation(portfolio_risk, idiosyncratic_risk, regime):
    """
    Dynamic allocation based on portfolio-level + idiosyncratic risk.
    - Higher risk → smaller weight, larger cash buffer
    """
    # Combined risk per ticker (50% portfolio + 50% idiosyncratic)
    combined_risk = {}
    for ticker in TICKERS:
        combined_risk[ticker] = round(
            portfolio_risk * 0.5 + idiosyncratic_risk[ticker] * 0.5, 4
        )

    # Base weights from inverse risk
    inv_risk = {t: 1 / (r + 0.01) for t, r in combined_risk.items()}
    total_inv = sum(inv_risk.values())
    base_weights = {t: v / total_inv for t, v in inv_risk.items()}

    # Cash buffer depends on regime
    if regime == 0:    # Normal
        cash_ratio = max(0, portfolio_risk * 0.2)
    elif regime == 1:  # Elevated
        cash_ratio = max(0.1, portfolio_risk * 0.4)
    else:              # Crisis
        cash_ratio = max(0.3, portfolio_risk * 0.6)

    # Allocate the remainder across tickers
    equity_ratio = 1 - cash_ratio
    final_weights = {t: w * equity_ratio for t, w in base_weights.items()}
    final_weights["CASH"] = cash_ratio

    return combined_risk, final_weights

combined_risk, weights = dynamic_allocation(
    portfolio_risk, idiosyncratic_risk, current_regime
)

print("\nCombined risk per ticker:")
for ticker, risk in combined_risk.items():
    print(f"  {ticker}: {risk:.4f}")

print("\nDynamic allocation result:")
print("-" * 40)
for asset, weight in sorted(weights.items(), key=lambda x: -x[1]):
    bar = "█" * int(weight * 30)
    print(f"  {asset:5s}: {weight*100:.1f}%  {bar}")

Current portfolio risk: 0.2190
Current regime: 1 (Elevated)

Combined risk per ticker:
  AAPL: 0.3226
  MSFT: 0.3010
  GOOGL: 0.2148
  JPM: 0.3261
  SOXX: 0.3595

Dynamic allocation result:
----------------------------------------
  GOOGL: 24.5%  ███████
  MSFT : 17.7%  █████
  AAPL : 16.5%  ████
  JPM  : 16.4%  ████
  SOXX : 14.9%  ████
  CASH : 10.0%  ███


In [15]:
# Save results
import json
from datetime import datetime

allocation_result = {
    "date": datetime.today().strftime("%Y-%m-%d"),
    "portfolio_risk": round(float(portfolio_risk), 4),
    "regime": int(current_regime),
    "idiosyncratic_risk": idiosyncratic_risk,
    "combined_risk": combined_risk,
    "weights": {k: round(v, 4) for k, v in weights.items()}
}

with open(f"{data_path}/allocation.json", "w") as f:
    json.dump(allocation_result, f, indent=2)

print("Saved: allocation.json")
print(json.dumps(allocation_result, indent=2))

Saved: allocation.json
{
  "date": "2026-07-13",
  "portfolio_risk": 0.219,
  "regime": 1,
  "idiosyncratic_risk": {
    "AAPL": 0.4261,
    "MSFT": 0.3831,
    "GOOGL": 0.2106,
    "JPM": 0.4332,
    "SOXX": 0.5
  },
  "combined_risk": {
    "AAPL": 0.3226,
    "MSFT": 0.301,
    "GOOGL": 0.2148,
    "JPM": 0.3261,
    "SOXX": 0.3595
  },
  "weights": {
    "AAPL": 0.1655,
    "MSFT": 0.177,
    "GOOGL": 0.2448,
    "JPM": 0.1638,
    "SOXX": 0.149,
    "CASH": 0.1
  }
}
